In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
#working env
import os
os.chdir(r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT')
os.listdir()

In [ ]:
adata = sc.read("integrated_harmony.h5ad")
adata.X=adata.layers['soupX_counts'].copy()
adata
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
sc.pl.umap(adata, color=["cell_type", "batch"])

In [ ]:
label_key = "cell_type"
batch_key = "batch"

In [ ]:
keep_batches = ["HFD"]
adata = adata[adata.obs[batch_key].isin(keep_batches)].copy()
adata

In [ ]:
adata_ms=adata[adata.obs["cell_type"] == "Ms"].copy()

In [ ]:
sc.pl.umap(adata_ms, color=["cell_type", "batch", "leiden_res0_5"])

In [ ]:
print(np.min(adata_ms.X))
print(np.max(adata_ms.X))


In [ ]:
adata

In [ ]:
sc.pp.normalize_total(adata_ms, target_sum=1e6)
sc.pp.log1p(adata_ms)
print(np.min(adata_ms.X))
print(np.max(adata_ms.X))
#sc.pp.highly_variable_genes(adata_ms)
#sc.pl.highly_variable_genes(adata_ms)
#adata_ms.raw = adata_ms
#adata_ms = adata_ms[:, adata_ms.var.highly_variable]
#sc.pp.regress_out(adata_ms, ['total_counts', 'pct_counts_mt'])
#sc.pp.scale(adata_ms, max_value=10)
#sc.tl.pca(adata_ms, svd_solver='arpack')
#sc.pl.pca(adata, color="batch")
#sc.pl.pca_variance_ratio(adata_ms, log=True)
#sc.pp.neighbors(adata_ms, n_neighbors=10, n_pcs=50)
#sc.tl.paga(adata_ms)
#sc.pl.paga(adata_ms, plot=False)  # remove `plot=False` if you want to see the coarse-grained graph
sc.pl.umap(adata_ms, color = "cell_type")



In [ ]:
adata_ms

In [ ]:
print(np.min(adata_ms.layers['counts']))
print(np.max(adata_ms.layers['counts']))

In [ ]:
#fig = plt.figure(figsize=(9, 11))
import matplotlib.pyplot as plt
tmp = pd.crosstab(adata.obs['leiden_res0_5'],adata.obs['batch'], normalize='columns').T.plot(kind='barh', stacked=True)
tmp.legend(title='Cell Types', bbox_to_anchor=(1.26, 1.02),loc='upper right')
plt.savefig("/Subcluster macro.pdf", bbox_inches = 'tight') # save as png

In [ ]:
sc.pp.neighbors(adata_ms, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata_ms)
sc.tl.leiden(adata_ms, key_added="leiden_res0_1", resolution=0.1)
sc.tl.leiden(adata_ms, key_added="leiden_res0_2", resolution=0.2)
sc.tl.leiden(adata_ms, key_added="leiden_res0_25", resolution=0.25)
sc.tl.leiden(adata_ms, key_added="leiden_res0_5",  resolution=0.5)
sc.tl.leiden(adata_ms, key_added="leiden_res1_0",  resolution=1.0)
sc.tl.leiden(adata_ms, key_added="leiden_res1_5",  resolution=1.5)

In [ ]:

sc.pl.umap(adata_ms,
        color=["leiden_res0_1", "leiden_res0_2", "leiden_res0_25", "leiden_res0_5", "leiden_res1_0", "leiden_res1_5", "n_genes_by_counts","pct_counts_mt", "batch", "Trem2"],
        legend_loc="on data",
        ncols = 3, legend_fontoutline=2
    )

In [ ]:
sc.tl.rank_genes_groups(adata_ms, 'leiden_res0_5', method='t-test')
sc.pl.rank_genes_groups(adata_ms, n_genes=25, sharey=False)

In [ ]:
from matplotlib.pyplot import rc_context

In [ ]:
adatatemp=adata_ms.copy()

In [ ]:
marker_genes_Wirka2019 = {
"TLF-Cd209hi":["Timd4","Lyve1","Folr2","Cd209d","Cd209f","Pf4","Cbr2","Maf"],
"TLF-Cd209low":["Timd4","Lyve1","Folr2","Pf4","Cbr2","Maf"],
"MacAir":["Acp5","Gngt2","Mpeg1","Cd72","H2-Eb1", "H2-Aa", "Cd74"],
"Trem2hi-Slamf9":["Trem2", "Slamf9", "Cd9", "Cd72"],
"Trem2hi-Gpnmb":["Acp5", "Gngt2","Mpeg1","Trem2","Cd9","Fabp5", "Gpnmb", "Ctsd"],
"Infl. Nlrp3":["Ctsd", "Cxcl12", "Ifrd1", "Nlrp3"],
"CCR2int-MHCII":["H2-Eb1", "H2-Aa", "Cd74"],
"IFNIC":["Isg15","Ifit3","Irf7"],
"Macro":["Adgre1","S100a6", "Ear2", "Plac8", "Msrb1"],
"NK/T cells": ["Cd3d","Cd8b1","Nkg7"],
"MHCIIhi":["Cd14", "Cst3", "Lilra5", "Apoe", "Trem2", "H3f3b", "C5ar1", "St3gal6", "Stab1"],
    "CCR2+": ["Ccr2", "Cd52", "H2-Eb1", "H2-Ab1", "S100a6", "Bcl2a1a", "S100a11", "Coro1a", "Cyp4f18", "Fxdy5"],
    "Zernecke2022-Macro subset": ["Retnla", "Cd226","Ccr2","Chil3" ,"Ly6c2", "Ace", "Treml4","Isg15", "Ifit1", "Irf7", "Oasl2", "Apoc1","Psap",  "Ctsd", "Lipa","Lpl", "Fth1", "Ftl1","Ctss", "Apoe","Fabp5", "Atp6v0d2", "Gpnmb", "Slamf9", "Cd72", "Ch25h", "Hexb", "Capg","Cd9","Itgax","Spp1","Trem2","Nlrp3", "Ifrd1", "Il1b", "Kdm6b","Tmem176b","Tmem176a","H2-Eb1", "H2-Aa", "Jun", "Fos", "Atf3","Stab1", "Pf4", "Cd163", "Maf","Cd209f", "Cd209d", "Cd209g", "Fxyd2", "Mrc1","Timd4", "Lyve1"],
    "Zernecke2022-DC subset": ["Cd80", "Cd86", "Cd40","Myo1g", "Cxcl16", "Adam8",
                                         "Relb", "Cd83", 
                                         "Socs1", "Socs2", "Cd274","Il4i1",
                                         "Cd200", "Ccr7", "Fscn1","Il12b","Tnfrsf4", "Ccl17", "Ccl22", "Cd209a", "Itgam", "Klrd1", "Clec10a", "Napsa","Xcr1","Clec9a","Naaa", "Irf8"],
    "Trem2+": ["Ctsd", "Cd9", "Lgals3"],
    "Foamy Wiliams 2020" : ["Fabp4, Ctsl, Atp6v0d2, Gpnmb, Fabp5, Htra1, Epb41l3, Pld3"],

}

In [ ]:
def sort_and_score(adatatemp, marker_genes):
    
    marker_genes_in_data = {}
    for ct, markers in marker_genes.items():
        markers_found = []
        for marker in markers:
            if marker in adatatemp.var.index:
                markers_found.append(marker)
        if markers_found:
            marker_genes_in_data[ct] = markers_found

    for key in marker_genes_in_data.keys():
        sc.tl.score_genes(adatatemp, marker_genes_in_data[key], score_name=key)
        
    return adatatemp, marker_genes_in_data

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Wirka2019)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    size=250, sort_order=True
    )

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False, cmap="Reds",
    )
    print("\n\n\n")  # print white space for legibility

In [ ]:
adata_HFD=adata_ms.copy()

In [ ]:
del adata
del adata_ms

In [ ]:
adata = sc.read("integrated_harmony.h5ad")
adata.X=adata.layers['soupX_counts'].copy()
adata
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
sc.pl.umap(adata, color=["cell_type", "batch"])

In [ ]:
label_key = "cell_type"
batch_key = "batch"

In [ ]:
keep_batches = ["HFD_L1"]
adata = adata[adata.obs[batch_key].isin(keep_batches)].copy()
adata

In [ ]:
adata_ms=adata[adata.obs["cell_type"] == "Ms"].copy()

In [ ]:
sc.pl.umap(adata_ms, color=["cell_type", "batch", "leiden_res0_5"])

In [ ]:
print(np.min(adata_ms.X))
print(np.max(adata_ms.X))


In [ ]:
adata

In [ ]:
sc.pp.normalize_total(adata_ms, target_sum=1e6)
sc.pp.log1p(adata_ms)
print(np.min(adata_ms.X))
print(np.max(adata_ms.X))
#sc.pp.highly_variable_genes(adata_ms)
#sc.pl.highly_variable_genes(adata_ms)
#adata_ms.raw = adata_ms
#adata_ms = adata_ms[:, adata_ms.var.highly_variable]
#sc.pp.regress_out(adata_ms, ['total_counts', 'pct_counts_mt'])
#sc.pp.scale(adata_ms, max_value=10)
#sc.tl.pca(adata_ms, svd_solver='arpack')
#sc.pl.pca(adata, color="batch")
#sc.pl.pca_variance_ratio(adata_ms, log=True)
#sc.pp.neighbors(adata_ms, n_neighbors=10, n_pcs=50)
#sc.tl.paga(adata_ms)
#sc.pl.paga(adata_ms, plot=False)  # remove `plot=False` if you want to see the coarse-grained graph
sc.pl.umap(adata_ms, color = "cell_type")



In [ ]:
adata_ms

In [ ]:
print(np.min(adata_ms.layers['counts']))
print(np.max(adata_ms.layers['counts']))

In [ ]:
sc.pp.neighbors(adata_ms, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata_ms)
sc.tl.leiden(adata_ms, key_added="leiden_res0_1", resolution=0.1)
sc.tl.leiden(adata_ms, key_added="leiden_res0_2", resolution=0.2)
sc.tl.leiden(adata_ms, key_added="leiden_res0_25", resolution=0.25)
sc.tl.leiden(adata_ms, key_added="leiden_res0_5",  resolution=0.5)
sc.tl.leiden(adata_ms, key_added="leiden_res1_0",  resolution=1.0)
sc.tl.leiden(adata_ms, key_added="leiden_res1_5",  resolution=1.5)

In [ ]:

sc.pl.umap(adata_ms,
        color=["leiden_res0_1", "leiden_res0_2", "leiden_res0_25", "leiden_res0_5", "leiden_res1_0", "leiden_res1_5", "n_genes_by_counts","pct_counts_mt", "batch", "Trem2"],
        legend_loc="on data",
        ncols = 3, legend_fontoutline=2
    )

In [ ]:
sc.tl.rank_genes_groups(adata_ms, 'leiden_res0_5', method='t-test')
sc.pl.rank_genes_groups(adata_ms, n_genes=25, sharey=False)

In [ ]:
from matplotlib.pyplot import rc_context

In [ ]:
adatatemp=adata_ms.copy()

In [ ]:
marker_genes_Wirka2019 = {
"TLF-Cd209hi":["Timd4","Lyve1","Folr2","Cd209d","Cd209f","Pf4","Cbr2","Maf"],
"TLF-Cd209low":["Timd4","Lyve1","Folr2","Pf4","Cbr2","Maf"],
"MacAir":["Acp5","Gngt2","Mpeg1","Cd72","H2-Eb1", "H2-Aa", "Cd74"],
"Trem2hi-Slamf9":["Trem2", "Slamf9", "Cd9", "Cd72"],
"Trem2hi-Gpnmb":["Acp5", "Gngt2","Mpeg1","Trem2","Cd9","Fabp5", "Gpnmb", "Ctsd"],
"Infl. Nlrp3":["Ctsd", "Cxcl12", "Ifrd1", "Nlrp3"],
"CCR2int-MHCII":["H2-Eb1", "H2-Aa", "Cd74"],
"IFNIC":["Isg15","Ifit3","Irf7"],
"Macro":["Adgre1","S100a6", "Ear2", "Plac8", "Msrb1"],
"NK/T cells": ["Cd3d","Cd8b1","Nkg7"],
"MHCIIhi":["Cd14", "Cst3", "Lilra5", "Apoe", "Trem2", "H3f3b", "C5ar1", "St3gal6", "Stab1"],
    "CCR2+": ["Ccr2", "Cd52", "H2-Eb1", "H2-Ab1", "S100a6", "Bcl2a1a", "S100a11", "Coro1a", "Cyp4f18", "Fxdy5"],
    "Zernecke2022-Macro subset": ["Retnla", "Cd226","Ccr2","Chil3" ,"Ly6c2", "Ace", "Treml4","Isg15", "Ifit1", "Irf7", "Oasl2", "Apoc1","Psap",  "Ctsd", "Lipa","Lpl", "Fth1", "Ftl1","Ctss", "Apoe","Fabp5", "Atp6v0d2", "Gpnmb", "Slamf9", "Cd72", "Ch25h", "Hexb", "Capg","Cd9","Itgax","Spp1","Trem2","Nlrp3", "Ifrd1", "Il1b", "Kdm6b","Tmem176b","Tmem176a","H2-Eb1", "H2-Aa", "Jun", "Fos", "Atf3","Stab1", "Pf4", "Cd163", "Maf","Cd209f", "Cd209d", "Cd209g", "Fxyd2", "Mrc1","Timd4", "Lyve1"],
    "Zernecke2022-DC subset": ["Cd80", "Cd86", "Cd40","Myo1g", "Cxcl16", "Adam8",
                                         "Relb", "Cd83", 
                                         "Socs1", "Socs2", "Cd274","Il4i1",
                                         "Cd200", "Ccr7", "Fscn1","Il12b","Tnfrsf4", "Ccl17", "Ccl22", "Cd209a", "Itgam", "Klrd1", "Clec10a", "Napsa","Xcr1","Clec9a","Naaa", "Irf8"],
    "Trem2+": ["Ctsd", "Cd9", "Lgals3"],
    "Foamy Wiliams 2020" : ["Fabp4, Ctsl, Atp6v0d2, Gpnmb, Fabp5, Htra1, Epb41l3, Pld3"],

}

In [ ]:
def sort_and_score(adatatemp, marker_genes):
    
    marker_genes_in_data = {}
    for ct, markers in marker_genes.items():
        markers_found = []
        for marker in markers:
            if marker in adatatemp.var.index:
                markers_found.append(marker)
        if markers_found:
            marker_genes_in_data[ct] = markers_found

    for key in marker_genes_in_data.keys():
        sc.tl.score_genes(adatatemp, marker_genes_in_data[key], score_name=key)
        
    return adatatemp, marker_genes_in_data

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Wirka2019)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    size=250, sort_order=True
    )

In [ ]:
for ct in marker_genes_in_data:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adatatemp,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax=0.9,
        #vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False, cmap="Reds",
    )
    print("\n\n\n")  # print white space for legibility

In [ ]:
adata_HFD_L1=adata_ms.copy()

In [ ]:
print("L1")
sc.pl.umap(adata_HFD_L1, color=["Trem2", "Ccr2", "Gpnmb", "Slamf9", "Fabp5", "leiden_res0_25"])
print("HFD")
sc.pl.umap(adata_HFD, color=["Trem2", "Ccr2", "Gpnmb", "Slamf9", "Fabp5"])

In [ ]:
#HFD
HFD_annotation = {
    "0": "Trem2-Gpnmb Ms",
    "1": "Trem2-Gpnmb Ms",
    "2": "Trem2-Gpnmb Ms",
    "3": "Trem2-Gpnmb Ms",
    "4": "NK/T-Cells",
    "5": "TLF-Ms",
   }

In [ ]:
adata_HFD.obs["cell_type"] = adata_HFD.obs.leiden_res0_5.map(HFD_annotation)

In [ ]:
sc.pl.umap(adata_HFD, color=["cell_type"])

In [ ]:
#HFD
HFD_L1_annotation = {
    "0": "Trem2-Gpnmb Ms",
    "1": "Trem2-Gpnmb Ms",
    "2": "Trem2-Gpnmb Ms",
    "3": "Trem2-Gpnmb Ms",
    "4": "Trem2-Gpnmb Ms",
    "5": "Trem2-Gpnmb Ms",
    "6": "Trem2-Gpnmb Ms",
    "7": "Trem2-Gpnmb Ms",
    "8": "TLF-Ms",
    "9": "NK/T-Cells"

   }

In [ ]:
adata_HFD_L1.obs["cell_type"] = adata_HFD_L1.obs.leiden_res1_0.map(HFD_L1_annotation)

In [ ]:
sc.pl.umap(adata_HFD_L1, color=["cell_type"])

In [ ]:
merged = adata_HFD.concatenate(adata_HFD_L1)

In [ ]:
merged.obs['batch'] = merged.obs['batch'].replace("0", "HFD")
merged.obs['batch'] = merged.obs['batch'].replace("1", "HFD_L1")

In [ ]:
np.max(adata_HFD_L1.layers["counts"])

In [ ]:
adata=merged.copy()

In [ ]:
merged.var["feature_types"].value_counts()

In [ ]:
adata = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()
sc.pp.filter_genes(adata, min_cells=1)
adata

In [ ]:
sc.pl.umap(adata, color=["batch", "cell_type"])

In [ ]:
adata.X = adata.layers["soupX_counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
np.max(adata.X)

In [ ]:
sc.pp.highly_variable_genes(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
adata

In [ ]:
np.max(adata.X)

In [ ]:
adata.uns[batch_key +   "_colors"] = [
    "#1b9e77",
    "#d95f02",

]  # Set custom colours for batches
sc.pl.umap(adata, color=[label_key, batch_key], wspace=1)

In [ ]:
data_integrated_harmony=adata.copy()

In [ ]:
import scanpy.external as sce

In [ ]:
sce.pp.harmony_integrate(data_integrated_harmony, "batch")

In [ ]:
data_integrated_harmony.obsm['X_pca'] = data_integrated_harmony.obsm['X_pca_harmony']
sc.pp.neighbors(data_integrated_harmony)
sc.tl.umap(data_integrated_harmony)
sc.tl.leiden(data_integrated_harmony, resolution=0.1)

In [ ]:
sc.pl.umap(data_integrated_harmony, color=[label_key, batch_key], wspace=1)

In [ ]:
data_integrated_harmony.write_h5ad("integrated_harmony_Ms_subset.h5ad")

In [ ]:
sc.tl.rank_genes_groups(adata, 'cell_type', method='t-test', key_added = "t-test")
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, key = "t-test")

In [ ]:
cl1 = adata[adata.obs['cell_type'] == 'Trem2-Gpnmb Ms',:]
cl1.obs['batch'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(cl1, 'batch', method='wilcoxon', key_added = "wilcoxon")
sc.pl.rank_genes_groups(cl1, n_genes=25, sharey=False, key="wilcoxon")

In [ ]:
cl1 = adata[adata.obs['cell_type'] == 'TLF-Ms',:]
cl1.obs['batch'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(cl1, 'batch', method='wilcoxon', key_added = "wilcoxon")
sc.pl.rank_genes_groups(cl1, n_genes=25, sharey=False, key="wilcoxon")

In [ ]:
cl1 = adata[adata.obs['cell_type'] == 'NK/T-Cells',:]
cl1.obs['batch'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(cl1, 'batch', method='wilcoxon', key_added = "wilcoxon")
sc.pl.rank_genes_groups(cl1, n_genes=25, sharey=False, key="wilcoxon")

In [ ]:
adata.write_h5ad("subclustered_Ms.h5ad")